# Machine Learning Algorithms 

## Library

In [1]:
import numpy as np
import pandas as pd

## Data Generation

In [ ]:
# Constantes de calibración (µT/V)
kx, ky, kz = 1.0, 1.0, 1.0  # suponemos lineales iguales

# Rango de voltajes
voltajes = np.arange(-10, 11, 1)  # -10 a 10 con paso de 1

# Todas las combinaciones
Vx, Vy, Vz = np.meshgrid(voltajes, voltajes, voltajes, indexing="ij")
Vx = Vx.flatten()
Vy = Vy.flatten()
Vz = Vz.flatten()

# Campos generados
Bx = kx * Vx
By = ky * Vy
Bz = kz * Vz

# Magnitud resultante
Bmag = np.sqrt(Bx**2 + By**2 + Bz**2)

# DataFrame
df = pd.DataFrame({
    "Vx": Vx,
    "Vy": Vy,
    "Vz": Vz,
    "Bx (µT)": Bx,
    "By (µT)": By,
    "Bz (µT)": Bz,
    "|B| (µT)": Bmag
})

# Guardar en CSV
df.to_csv("datos_helmholtz_-10a10.csv", index=False)

print("CSV generado: datos_helmholtz_-10a10.csv")


CSV generado: datos_helmholtz_-10a10.csv


## Linear Regression

In [2]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# Cargar el dataset (el que generaste antes)
df = pd.read_csv("Training_data.csv")

# Variables de entrada: campos
X = df[["Bx (µT)", "By (µT)", "Bz (µT)"]]

# Variables de salida: voltajes
y = df[["Vx", "Vy", "Vz"]]

# Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Crear y entrenar el modelo
model = LinearRegression()
model.fit(X_train, y_train)

# Evaluar el modelo
score = model.score(X_test, y_test)
print("R²:", score)

# Ejemplo de predicción
campo = pd.DataFrame([[5, -3, 2]], columns=["Bx (µT)", "By (µT)", "Bz (µT)"])  # Bx=5, By=-3, Bz=2 µT
voltajes_pred = model.predict(campo)
print("Voltajes predichos:", voltajes_pred)


R²: 0.9994773468528827
Voltajes predichos: [[-0.18502362  2.72646804  1.84752375]]


## Polynomial Regression

In [3]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

# Cargar el dataset
df = pd.read_csv("Training_data.csv")

# Features: campos magnéticos
X = df[["Bx (µT)", "By (µT)", "Bz (µT)"]]

# Target: voltajes
y = df[["Vx", "Vy", "Vz"]]

# Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Pipeline con polinomios de grado 2
poly_model = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),  # puedes subir a degree=3 si lo necesitas
    ("linreg", LinearRegression())
])

# Entrenar
poly_model.fit(X_train, y_train)

# Evaluación
score = poly_model.score(X_test, y_test)
print("R² Polynomial Regression:", score)

# Ejemplo de predicción
campo = pd.DataFrame([[5, -3, 2]], columns=["Bx (µT)", "By (µT)", "Bz (µT)"])
print("Predicción voltajes:", poly_model.predict(campo))


R² Polynomial Regression: 0.9995945426293303
Predicción voltajes: [[-0.25864813  2.70031509  1.81220553]]


## Random Forest Regressor

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# Cargar el dataset
df = pd.read_csv("Training_data.csv")

# Features: campos magnéticos
X = df[["Bx (µT)", "By (µT)", "Bz (µT)"]]

# Target: voltajes
y = df[["Vx", "Vy", "Vz"]]

# Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Crear el modelo (multi-output)
rf = RandomForestRegressor(
    n_estimators=200,   # número de árboles
    max_depth=None,     # sin límite de profundidad
    random_state=42,
    n_jobs=-1
)

# Entrenar
rf.fit(X_train, y_train)

# Evaluar
score = rf.score(X_test, y_test)
print("R² Random Forest:", score)

# Ejemplo de predicción
campo = pd.DataFrame([[5, -3, 2]], columns=["Bx (µT)", "By (µT)", "Bz (µT)"])
print("Predicción voltajes:", rf.predict(campo))


R² Random Forest: 0.9988541192783088
Predicción voltajes: [[-0.235  3.1    1.78 ]]


## Neural Networks

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Cargar el dataset
df = pd.read_csv("Training_data.csv")

# Features: campos magnéticos
X = df[["Bx (µT)", "By (µT)", "Bz (µT)"]]

# Target: voltajes
y = df[["Vx", "Vy", "Vz"]]

# Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Definir pipeline con normalización + red neuronal
mlp = Pipeline([
    ("scaler", StandardScaler()),  # escalar datos para ayudar al entrenamiento
    ("nn", MLPRegressor(
        hidden_layer_sizes=(64, 64),  # dos capas ocultas de 64 neuronas cada una
        activation="relu",            # función de activación
        solver="adam",                # optimizador
        max_iter=1000,                # número máximo de iteraciones
        random_state=42
    ))
])

# Entrenar
mlp.fit(X_train, y_train)

# Evaluar
score = mlp.score(X_test, y_test)
print("R² Neural Network:", score)

# Ejemplo de predicción
campo = pd.DataFrame([[5, -3, 2]], columns=["Bx (µT)", "By (µT)", "Bz (µT)"])
print("Predicción voltajes:", mlp.predict(campo))


R² Neural Network: 0.9996484922441612
Predicción voltajes: [[-0.2510383   2.7273787   1.80084958]]
